Given an array of integers, find the length of the longest strictly increasing subsequence.  
Example :- Input : nums : {10, 9, 2, 5, 3, 7, 101, 18} -> Output : 4, (subsequence {2,5,7,101} or {2,3,7,101}).

## Recursive Implementation
This can be solved with exhaustive enumeration, at each recursion we have a choice of picking or not picking the number. The slate will have the running length and the index of last max value(to check if the current value is greater than the last max value in that path).

```C++
int max_length = 0;
void longest_increasing_subsequence(vector<int>& nums, int i, int last_max_i, int length)
{
    if(i == nums.size())
    {
        max_length = max(length, max_length);
        return;
    }

    //Include ith element
    int new_length = length;
    int new_last_max_i = last_max_i;
    if(last_max_i == -1 || nums[i] > nums[last_max_i])
    {
        new_length++;
        new_last_max_i = i;
    }
    longest_increasing_subsequence(nums, i+1, new_last_max_i, new_length);

    //Not include ith element
    longest_increasing_subsequence(nums, i+1, last_max_i, length);
}

int lengthOfLIS(vector<int>& nums) 
{
    longest_increasing_subsequence(nums, 0, -1, 0);
    return max_length;
}
```
The time complexity for this algorithm is O(2<sup>n</sup>) as we have the option to choose or not choose each element, space complexity is O(n) for the call stack. 

For a recursion function to be converted into a DP solution, should the recursive function always return? To study, below is the return recursive function.

```C++
int longest_increasing_subsequence(vector<int>& nums, int i, int last_max_i)
{
    if(i == nums.size())
    {
        return 0;
    }

    if(last_max_i != -1 && dp[i][last_max_i] != -1)
    {
        return dp[i][last_max_i];
    }

    //Not include ith element
    int not_take = longest_increasing_subsequence(nums, i+1, last_max_i);

    //Include ith element
    int take = 0;
    if(last_max_i == -1 || nums[i] > nums[last_max_i])
    {
        take = 1 + longest_increasing_subsequence(nums, i+1, i);
    }

    dp[i][last_max_i] = max(not_take, take);
    return dp[i][last_max_i];
}
```

## Bottom-up DP solution
Can we use DP here? We are solving the same subproblems on different tree branches by making different decisions in the past, i.e we can land on the same number coming from different paths, the solution from that number onwards will be the same, hence it can be solved with DP.
* Data structure : i and last_max_i are changing, so we need a 2D array for DP.
* Size : We are reaching the max length of nums from 0, so we need n+1 size in both the dimensions.
* Direction : Since f(i, last_max_i) is dependent on f(i+1, i/last_max_i) we move from high to low, i -> n-1 to 0, last_max_i from i-1 to -1.
* Copy paste logic from recursion

```C++
int lengthOfLIS(vector<int>& nums) 
{
    //Create a dp of size n+1 x n+1 and intialize it with 0
    //Base condition dp[n][n] = 0
    vector<vector<int>> dp;
    for(int i = 0; i <= nums.size(); i++)
    {
        vector<int> cols;
        cols.resize(nums.size() + 1, 0);

        dp.push_back(cols);
    }

    //Fill up the DP in the reverse direction
    for(int i = nums.size()-1; i >= 0; i--) //i=n is the base condition, filled with 0
    {
        for(int prev_i = i -1; prev_i >= -1; prev_i--)
        {
            int not_take = dp[i+1][prev_i+1];
            int take = 0;
            if(prev_i == -1 || nums[prev_i] < nums[i])
            {
                take = 1 + dp[i+1][i+1];
            }
            dp[i][prev_i+1] = max(take, not_take);
        }
    }

    return dp[0][0];
}
```

Looks like we can convert the first recursive implementation above to DP, by moving in the forward direction and only having 1D array. To study.

```C++
int lengthOfLIS(vector<int>& nums) 
{
    //Create a 1D dp of size n
    vector<int> dp;
    dp.resize(nums.size(), 1);

    //Base condition, first item will be of length 1
    dp[0] = 1;
    //Variable to hold the max length
    int max_length = 1;

    //Fill up the DP in the forward direction
    for(int i = 1; i < nums.size(); i++) 
    {
        for(int prev_i = 0; prev_i < i; prev_i++)
        {
            if(nums[prev_i] < nums[i])
            {
                //May pick or not pick
                dp[i] = max(dp[i], 1 + dp[prev_i]);  
            }
            max_length = max(max_length, dp[i]);
        }
    }

    return max_length;
}
```
Time complexity now is O(n<sup>2</sup>) now to fill up the 2D array and the space complexity is also O(n<sup>2</sup>).